In [26]:
%pip install numpy pandas scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
import sqlite3
from collections import defaultdict
from html import escape
from urllib.parse import quote

from IPython.display import HTML

import numpy as np
import pandas as pd
from scipy import sparse

DB_PATH = "favourites.sqlite"
# artists with fewer fans than this have too little overlap to say who they're like
MIN_FANS = 3
# Danbooru files voice actors of animated posts under artist tags
SKIP_VOICE_ACTORS = True
# share of each user's favourites, newest first, hidden in the test
HELD_OUT = 0.2
TEST_DEPTHS = [10, 50, 200]
# how hard popular artists are pushed down: 0.5 is plain cosine, higher favours artists with fewer fans
# POPULARITY_DAMPING = 0.75
POPULARITY_DAMPING = 0.5
# a link resting on few shared fans keeps shared / (shared + SHRINK) of its score, see ARTISTS.md
SHARED_FANS_SHRINK = 5.0
# how much less users who are fans of many artists count: 0 counts everyone the same, 1 divides by their artist count
# HEAVY_USER_DAMPING = 0.0
HEAVY_USER_DAMPING = 0.5
# how much what artists draw counts against who their fans are: 0 is fans only, 1 is tags only
TAG_WEIGHT = 0.0
# the (popularity, heavy users, tag weight) settings the test compares
TEST_SETTINGS = [
  (0.5, 0.0, 0.0),
  (0.5, 0.5, 0.0),
  (0.75, 0.5, 0.0),
  (0.75, 0.5, 0.25),
  (0.75, 0.5, 0.5),
  (0.75, 0.5, 0.75),
  (0.75, 0.5, 1.0),
]

In [28]:
db = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
rows = db.execute(
  """SELECT users.name, favourites.position, posts.artists FROM favourites
     JOIN users ON users.id = favourites.user_id
     JOIN posts ON posts.id = favourites.post_id
     ORDER BY users.name, favourites.position"""
).fetchall()
db.close()

# each user's favourites as lists of artists, newest favourite first
favourites_by_user: dict[str, list[list[str]]] = defaultdict(list)
for user, _, artists in rows:
  names = [name for name in artists.split() if not (SKIP_VOICE_ACTORS and name.endswith("_(voice_actor)"))]
  favourites_by_user[user].append(names)

print(f"{len(favourites_by_user)} users, {len(rows)} favourites")

506 users, 1449370 favourites


In [29]:
def fan_matrix(artist_sets: list[set[str]]) -> tuple[sparse.csc_matrix, list[str], dict[str, int]]:
  """A users x artists 0/1 matrix, keeping only artists with at least MIN_FANS fans."""
  fans: dict[str, int] = defaultdict(int)
  for artists in artist_sets:
    for artist in artists:
      fans[artist] += 1
  names = sorted(artist for artist, count in fans.items() if count >= MIN_FANS)
  column_of = {artist: column for column, artist in enumerate(names)}
  user_rows, columns = [], []
  for row, artists in enumerate(artist_sets):
    for artist in artists:
      if artist in column_of:
        user_rows.append(row)
        columns.append(column_of[artist])
  matrix = sparse.csc_matrix(
    (np.ones(len(columns)), (user_rows, columns)), shape=(len(artist_sets), len(names))
  )
  return matrix, names, column_of


def artists_of(favourites: list[list[str]]) -> set[str]:
  return {artist for post in favourites for artist in post}


users = sorted(favourites_by_user)
matrix, names, column_of = fan_matrix([artists_of(favourites_by_user[user]) for user in users])
fan_counts = np.asarray(matrix.sum(axis=0)).ravel()
print(f"{len(names)} artists with at least {MIN_FANS} fans")

54686 artists with at least 3 fans


In [30]:
db = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
post_rows = db.execute("SELECT id, artists, copyrights, characters, general FROM posts").fetchall()
# each user's favourite post ids, newest first, in the same order as favourites_by_user
favourite_posts: dict[str, list[int]] = defaultdict(list)
for user, post_id in db.execute(
  """SELECT users.name, favourites.post_id FROM favourites JOIN users ON users.id = favourites.user_id
     ORDER BY users.name, favourites.position"""
):
  favourite_posts[user].append(post_id)
db.close()

# profile rows follow the full fan matrix's artists; the test's smaller matrices look their artists up here
profile_row_of = dict(column_of)
post_index_of: dict[int, int] = {}
tag_column_of: dict[str, int] = {}
artist_post_pairs: tuple[list[int], list[int]] = ([], [])
post_tag_pairs: tuple[list[int], list[int]] = ([], [])
for post_id, artists, copyrights, characters, general in post_rows:
  post_artists = [profile_row_of[artist] for artist in artists.split() if artist in profile_row_of]
  if not post_artists:
    continue
  post = post_index_of.setdefault(post_id, len(post_index_of))
  for artist in post_artists:
    artist_post_pairs[0].append(artist)
    artist_post_pairs[1].append(post)
  for tag in f"{copyrights} {characters} {general}".split():
    post_tag_pairs[0].append(post)
    post_tag_pairs[1].append(tag_column_of.setdefault(tag, len(tag_column_of)))
del post_rows

artist_posts = sparse.csr_matrix(
  (np.ones(len(artist_post_pairs[0])), artist_post_pairs), shape=(len(profile_row_of), len(post_index_of))
)
post_tags = sparse.csr_matrix((np.ones(len(post_tag_pairs[0])), post_tag_pairs), shape=(len(post_index_of), len(tag_column_of)))
tag_counts = (artist_posts @ post_tags).tocsr()
posts_per_artist = np.asarray(artist_posts.sum(axis=1)).ravel()


artists_with_tag = np.asarray((tag_counts > 0).sum(axis=0)).ravel()
# tags most artists use count for little, a series or character used by few counts a lot
tag_rarity = sparse.diags(np.log((1 + tag_counts.shape[0]) / (1 + artists_with_tag)))


def tag_profiles(tag_counts: sparse.csr_matrix, posts_per_artist: np.ndarray) -> sparse.csr_matrix:
  """Each artist's share of posts with each tag, weighted by tag rarity, scaled to length 1 so dot products are cosines."""
  shares = sparse.diags(1 / np.maximum(posts_per_artist, 1)) @ tag_counts
  weighted = (shares @ tag_rarity).tocsr()
  lengths = np.sqrt(np.asarray(weighted.multiply(weighted).sum(axis=1)).ravel())
  return (sparse.diags(1 / np.maximum(lengths, 1e-12)) @ weighted).tocsr()


artist_profiles = tag_profiles(tag_counts, posts_per_artist)
print(f"{len(post_index_of)} posts, {len(tag_column_of)} tags, profiles for {len(profile_row_of)} artists")

870168 posts, 137600 tags, profiles for 54686 artists


In [31]:
def recommend(
  liked: list[str],
  matrix: sparse.csc_matrix,
  names: list[str],
  column_of: dict[str, int],
  count: int = 20,
  popularity: float | None = None,
  heavy_users: float | None = None,
  tag_weight: float | None = None,
  profile_changes: sparse.csr_matrix | None = None,
) -> pd.DataFrame:
  # read here rather than as defaults, so changing the settings cell takes effect without rerunning this one
  popularity = POPULARITY_DAMPING if popularity is None else popularity
  heavy_users = HEAVY_USER_DAMPING if heavy_users is None else heavy_users
  tag_weight = TAG_WEIGHT if tag_weight is None else tag_weight
  seeds = [column_of[artist] for artist in liked if artist in column_of]
  if not seeds:
    return pd.DataFrame(columns=["artist", "score", "fans", "closest liked artist"])
  fans = np.asarray(matrix.sum(axis=0)).ravel()
  artists_per_user = np.asarray(matrix.sum(axis=1)).ravel()
  # each user's weight: how many of the liked artists they're a fan of, each scaled by that artist's size
  user_weights = matrix[:, seeds] @ (1 / np.sqrt(np.maximum(fans[seeds], 1)))
  user_weights = user_weights / np.maximum(artists_per_user, 1) ** heavy_users
  scores = (matrix.T @ user_weights) / np.maximum(fans, 1) ** popularity
  # people who are fans of both the candidate and a liked artist; few of them is a coincidence, not a likeness
  liked_fans = (np.asarray(matrix[:, seeds].sum(axis=1)).ravel() > 0).astype(float)
  shared = matrix.T @ liked_fans
  scores = scores * shared / (shared + SHARED_FANS_SHRINK)
  if tag_weight > 0:
    rows = np.array([profile_row_of[name] for name in names])
    liked_profile = np.asarray(artist_profiles[rows[seeds]].sum(axis=0)).ravel()
    if profile_changes is not None:
      liked_profile += np.asarray(profile_changes[rows[seeds]].sum(axis=0)).ravel()
    every_score = artist_profiles @ liked_profile
    if profile_changes is not None:
      every_score += profile_changes @ liked_profile
    tag_scores = every_score[rows]
    # the two scores are on different scales, so each is divided by its best before blending
    scores = (1 - tag_weight) * scores / max(scores.max(), 1e-12) + tag_weight * tag_scores / max(tag_scores.max(), 1e-12)
  scores[seeds] = -np.inf
  top = np.argsort(-scores)[:count]
  top = top[scores[top] > 0]
  # the liked artist each recommendation overlaps with most, as a reason
  overlaps = (matrix[:, seeds].T @ matrix[:, top]).toarray()
  closest = [names[seeds[index]] for index in overlaps.argmax(axis=0)]
  return pd.DataFrame(
    {"artist": [names[column] for column in top], "score": scores[top].round(3), "fans": fans[top].astype(int), "closest liked artist": closest}
  )


GELBOORU_SEARCH = "https://gelbooru.com/index.php?page=post&s=list&tags="


def with_links(table: pd.DataFrame, columns: tuple[str, ...] = ("artist", "closest liked artist")):
  """Shows the table with artist names linking to their posts on Gelbooru."""
  def linked(artist: str) -> str:
    if not artist:
      return ""
    return f'<a href="{GELBOORU_SEARCH}{quote(artist)}" target="_blank">{escape(artist)}</a>'
  shown = table.copy()
  for column in columns:
    if column in shown:
      shown[column] = shown[column].fillna("").map(linked)
  return HTML(shown.to_html(escape=False, index=False, na_rep=""))

### From artists you pick

Put in Danbooru artist tags. Ones with under `MIN_FANS` fans are ignored.

In [32]:
# my_artists = ["shefu", "null_(nyanpyoun)"]
my_artists = ["kusakihara_toshiyuki"]
with_links(recommend(my_artists, matrix, names, column_of))

artist,score,fans,closest liked artist
gaby_(kaswooshle),0.013,3,kusakihara_toshiyuki
pokomi_9,0.013,4,kusakihara_toshiyuki
d4ggerfish,0.013,3,kusakihara_toshiyuki
mochi2machi,0.013,3,kusakihara_toshiyuki
uekura_eku,0.012,3,kusakihara_toshiyuki
sayosato,0.011,3,kusakihara_toshiyuki
toshi_(toshi10416),0.011,5,kusakihara_toshiyuki
n1ngenno_kuzu,0.010,4,kusakihara_toshiyuki
bryce_carrington,0.010,3,kusakihara_toshiyuki
luce-in-the-sky,0.010,3,kusakihara_toshiyuki


### From a Gelbooru profile

Gelbooru only shows favourites to the API, so this needs your API key and user id from
[Gelbooru's account options](https://gelbooru.com/index.php?page=account&s=options). Put them in
`gelbooru-credentials.json` next to this notebook (it's gitignored):

```json
{"api_key": "...", "user_id": "..."}
```

The profile can be anyone's, as a profile link or just the id. Gelbooru tags don't say which are artists, so a tag
counts as an artist when it's one of the Danbooru artists in the scraped data, which are the only ones the
recommender knows anyway.

In [33]:
# import json
# import pathlib
# import re
# import time
# import urllib.parse
# import urllib.request
# from html import unescape

# GELBOORU_PROFILE = "https://gelbooru.com/index.php?page=account&s=profile&id=1"
# GELBOORU_API = "https://gelbooru.com/index.php"
# GELBOORU_PAGE_SIZE = 100
# # Gelbooru refuses pages past 20,000 posts into a search
# GELBOORU_MAX_PAGES = 200
# # the app keeps to 2 requests a second too
# GELBOORU_PAUSE = 0.5


# def gelbooru_favourite_tags(profile: str) -> list[list[str]]:
#   """The tags of every post the profile favourited, read through the fav:<id> search."""
#   user_id = re.search(r"\bid=(\d+)", profile) or re.fullmatch(r"\s*(\d+)\s*", profile)
#   if not user_id:
#     raise ValueError(f"no user id in {profile!r}; use a profile link with id= or just the number")
#   credentials = json.loads(pathlib.Path("gelbooru-credentials.json").read_text(encoding="utf-8"))
#   posts: list[list[str]] = []
#   for page in range(GELBOORU_MAX_PAGES):
#     query = urllib.parse.urlencode({
#       "page": "dapi", "s": "post", "q": "index", "json": 1, "limit": GELBOORU_PAGE_SIZE, "pid": page,
#       "tags": f"fav:{user_id.group(1)}", "api_key": credentials["api_key"], "user_id": credentials["user_id"],
#     })
#     request = urllib.request.Request(f"{GELBOORU_API}?{query}", headers={"User-Agent": "foryooru-favourites/1.0"})
#     with urllib.request.urlopen(request, timeout=60) as response:
#       batch = json.load(response).get("post", [])
#     # tags come HTML-escaped, like &amp; in names with &
#     posts.extend(unescape(post["tags"]).split() for post in batch)
#     if len(batch) < GELBOORU_PAGE_SIZE:
#       break
#     time.sleep(GELBOORU_PAUSE)
#   return posts


# gelbooru_posts = gelbooru_favourite_tags(GELBOORU_PROFILE)
# gelbooru_fans = defaultdict(int)
# for tags in gelbooru_posts:
#   for tag in tags:
#     if tag in column_of:
#       gelbooru_fans[tag] += 1
# print(f"{len(gelbooru_posts)} favourites, {len(gelbooru_fans)} known artists among them")
# print("most favourited:", ", ".join(f"{artist} ({count})" for artist, count in sorted(gelbooru_fans.items(), key=lambda item: -item[1])[:10]))
# with_links(recommend(sorted(gelbooru_fans), matrix, names, column_of))

### Artists who draw a tag

Artists for whom a tag is a signature: ranked by the tag's weight in their profile, so an artist who draws it in most
posts beats one who has a lot of posts and a few with the tag.

In [37]:
def artists_drawing(tag: str, count: int = 20) -> pd.DataFrame:
  column = tag_column_of[tag]
  weights = artist_profiles[:, column].toarray().ravel()
  top = np.argsort(-weights)[:count]
  top = top[weights[top] > 0]
  with_tag = tag_counts[top, column].toarray().ravel().astype(int)
  return pd.DataFrame({
    "artist": [names[row] for row in top],
    "score": weights[top].round(3),
    "posts with the tag": with_tag,
    "posts": posts_per_artist[top].astype(int),
    "fans": fan_counts[top].astype(int),
  })


with_links(artists_drawing("thigh_strap"))

artist,score,posts with the tag,posts,fans
m.k_(mr.k),0.201,8,12,6
pxax3333,0.191,2,2,3
hiiragi_(hr_ccr),0.183,10,20,12
wakya,0.167,4,6,5
yuusa_matsushima,0.157,2,3,4
d3xrk,0.155,1,1,4
kyuuhyaku,0.145,7,9,6
07touka25,0.145,2,3,3
sinemi,0.144,6,7,11
pochadon,0.133,2,4,4


### Related tags

People who favourite posts with one tag also favourite which others? For each user, the share of their favourites
that have each tag; two tags are related when their shares go up and down together across users (a correlation). Tags
that everyone has a lot of, like `1girl`, don't move with anything in particular, so they don't show up as related to
everything. Only tags on at least `RELATED_MIN_POSTS` favourited posts are compared.

In [19]:
RELATED_MIN_POSTS = 200

# which scraped posts each user favourited, as a users x posts matrix
user_post_pairs: tuple[list[int], list[int]] = ([], [])
for row, user in enumerate(users):
  for post_id in favourite_posts[user]:
    if post_id in post_index_of:
      user_post_pairs[0].append(row)
      user_post_pairs[1].append(post_index_of[post_id])
user_posts = sparse.csr_matrix((np.ones(len(user_post_pairs[0])), user_post_pairs), shape=(len(users), len(post_index_of)))

tag_names = list(tag_column_of)
common_tags = np.flatnonzero(np.asarray(post_tags.sum(axis=0)).ravel() >= RELATED_MIN_POSTS)
favourites_per_user = np.maximum(np.asarray(user_posts.sum(axis=1)).ravel(), 1)
tag_shares = (sparse.diags(1 / favourites_per_user) @ (user_posts @ post_tags[:, common_tags])).toarray()
# centred and scaled per tag, so a dot product between two tags is their correlation across users
tag_shares -= tag_shares.mean(axis=0)
tag_shares /= np.maximum(np.linalg.norm(tag_shares, axis=0), 1e-12)
common_position = {tag_names[column]: position for position, column in enumerate(common_tags)}


def related_tags(tag: str, count: int = 20) -> pd.DataFrame:
  if tag not in common_position:
    raise ValueError(f"{tag} is on fewer than {RELATED_MIN_POSTS} favourited posts")
  correlations = tag_shares.T @ tag_shares[:, common_position[tag]]
  top = [position for position in np.argsort(-correlations) if position != common_position[tag]][:count]
  return pd.DataFrame({
    "tag": [tag_names[common_tags[position]] for position in top],
    "correlation": correlations[top].round(2),
  })


print(f"{len(common_tags)} tags compared over {len(users)} users")
related_tags("glasses")

7610 tags compared over 430 users


,tag,correlation
0,black-framed_eyewear,0.56
1,round_eyewear,0.51
2,opaque_glasses,0.39
3,real_life,0.33
4,yellow_neckerchief,0.32
5,heart_hands_duo,0.31
6,mojyo,0.30
7,photo_background,0.30
8,pillbox_hat,0.30
9,semi-rimless_eyewear,0.29


### For one scraped user

Recommends from the user's older favourites, then marks the artists they went on to favourite in their newest ones.
The user's newest favourites are taken out of the matrix first, so the answer can't leak in.

In [20]:
def split(favourites: list[list[str]]) -> tuple[list[list[str]], list[list[str]]]:
  held_count = round(len(favourites) * HELD_OUT)
  return favourites[held_count:], favourites[:held_count]


def without_newest(user: str) -> tuple[sparse.csc_matrix, list[str], dict[str, int], set[str], set[str]]:
  """The fan matrix with this user's newest favourites hidden, their earlier artists, and the new ones they went on to like."""
  earlier, held = split(favourites_by_user[user])
  known = artists_of(earlier)
  went_on_to_like = artists_of(held) - known
  # swap this user's row for their earlier artists; artists left the full matrix out have too few fans either way
  row = users.index(user)
  others = sparse.diags((np.arange(len(users)) != row).astype(float)) @ matrix
  known_columns = [column_of[artist] for artist in known if artist in column_of]
  own_row = sparse.csr_matrix((np.ones(len(known_columns)), ([row] * len(known_columns), known_columns)), shape=matrix.shape)
  hidden = (others + own_row).tocsc()
  # hiding favourites can drop artists below MIN_FANS, and they leave like they would in a rebuild
  kept = np.flatnonzero(np.asarray(hidden.sum(axis=0)).ravel() >= MIN_FANS)
  kept_names = [names[column] for column in kept]
  return hidden[:, kept], kept_names, {artist: column for column, artist in enumerate(kept_names)}, known, went_on_to_like


example_user = max(users, key=lambda user: len(favourites_by_user[user]))
test_matrix, test_names, test_column_of, known, went_on_to_like = without_newest(example_user)
picks = recommend(sorted(known), test_matrix, test_names, test_column_of, count=30)
picks["went on to like"] = picks["artist"].isin(went_on_to_like).map({True: "yes", False: ""})
print(f"{example_user}: {len(known)} earlier artists, {len(went_on_to_like)} new artists in the newest favourites")
with_links(picks)

2ampm: 1768 earlier artists, 317 new artists in the newest favourites


artist,score,fans,closest liked artist,went on to like
hood_(james_x),1.555,141,ebora,
distr,1.546,158,ebora,
hews,1.544,219,ebora,
rororo,1.544,189,ebora,
usa37107692,1.541,85,ebora,
teddypocky,1.539,119,ebora,yes
nvl,1.528,126,ebora,
hirosenpai,1.526,78,ebora,
oroborus,1.526,146,melowh,
reia_76,1.522,116,ebora,


## What niche and mainstream users like

How niche a user is: the median fan count of the artists they favourite, counting every artist, not just the ones with
`MIN_FANS`. "Only fan" is the share of their artists nobody else in the data favourites. Users with fewer than
`TASTE_MIN_FAVOURITES` favourites are left out, since a handful of favourites says little about their taste.

In [21]:
TASTE_MIN_FAVOURITES = 100

all_fans: dict[str, int] = defaultdict(int)
for user in users:
  for artist in artists_of(favourites_by_user[user]):
    all_fans[artist] += 1


def taste_shape(user: str) -> dict:
  fan_counts_of_theirs = [all_fans[artist] for artist in artists_of(favourites_by_user[user])]
  return {
    "user": user,
    "favourites": len(favourites_by_user[user]),
    "artists": len(fan_counts_of_theirs),
    "median fans": float(np.median(fan_counts_of_theirs)),
    "only fan": round(sum(count == 1 for count in fan_counts_of_theirs) / len(fan_counts_of_theirs), 2),
  }


shapes = pd.DataFrame([taste_shape(user) for user in users if len(favourites_by_user[user]) >= TASTE_MIN_FAVOURITES])
shapes = shapes.sort_values(["median fans", "only fan"], ascending=[True, False]).reset_index(drop=True)
shapes

,user,favourites,artists,median fans,only fan
0,user000727,1909,340,1.0,0.52
1,SidBarrett,454,389,2.0,0.49
2,kwsn,1414,625,2.0,0.48
3,BrYAlan,107,69,2.0,0.43
4,Nerniply,2303,1029,2.0,0.41
...,...,...,...,...,...
391,Drakengard811,1151,367,52.0,0.00
392,Call_Me_Mommy,751,353,53.0,0.01
393,Karkade,838,289,61.0,0.00
394,Doroo7,1462,219,62.0,0.00


## Test over every user

For each user, hide their newest favourites, recommend from the rest, and count how many of the new artists they
went on to favourite appear in the top 10 / 50 / 200. Only new artists that have at least `MIN_FANS` fans among the
other users are counted, since nothing could recommend the rest. The baseline suggests the most popular artists the user
doesn't know yet. Every user counts the same in the averages.

The test runs each of `TEST_SETTINGS`. For settings that use tags, the user's hidden posts are taken out of the tag
profiles too, so a new artist's profile isn't built from the posts being looked for. "Median fans in top 10" shows how
popular the picks are, so a setting that finds as many new artists with fewer fans is the one that discovers more.

In [22]:
def popular(matrix: sparse.csc_matrix, names: list[str], known: set[str], count: int) -> list[str]:
  fans = np.asarray(matrix.sum(axis=0)).ravel()
  ranked = [names[column] for column in np.argsort(-fans)]
  return [artist for artist in ranked if artist not in known][:count]


def found(picks: list[str], findable: set[str]) -> dict[str, float]:
  return {f"top {depth}": len(findable & set(picks[:depth])) / len(findable) for depth in TEST_DEPTHS}


def setting_name(popularity: float, heavy_users: float, tag_weight: float) -> str:
  return f"popularity {popularity}, heavy users {heavy_users}" + (f", tags {tag_weight}" if tag_weight else "")


def profile_changes_without_newest(user: str) -> sparse.csr_matrix:
  """How the tag profiles change without the posts this user hid, so a new artist's profile isn't built from the answer.

  Only the artists of the hidden posts change, so only their rows are rebuilt; every other row stays zero.
  """
  held_count = round(len(favourite_posts[user]) * HELD_OUT)
  held = [post_index_of[post_id] for post_id in favourite_posts[user][:held_count] if post_id in post_index_of]
  hidden_posts = artist_posts[:, held].tocsc()
  affected = np.unique(hidden_posts.nonzero()[0])
  counts = tag_counts[affected] - hidden_posts[affected] @ post_tags[held]
  counts.eliminate_zeros()
  rebuilt = tag_profiles(counts.tocsr(), posts_per_artist[affected] - np.asarray(hidden_posts[affected].sum(axis=1)).ravel())
  changes = (rebuilt - artist_profiles[affected]).tocoo()
  return sparse.csr_matrix((changes.data, (affected[changes.row], changes.col)), shape=artist_profiles.shape)


uses_tags = any(tag_weight > 0 for _, _, tag_weight in TEST_SETTINGS)
results = []
for user in users:
  test_matrix, test_names, test_column_of, known, went_on_to_like = without_newest(user)
  findable = {artist for artist in went_on_to_like if artist in test_column_of}
  if not findable or not known & test_column_of.keys():
    continue
  test_fans = np.asarray(test_matrix.sum(axis=0)).ravel()
  baseline = popular(test_matrix, test_names, known, max(TEST_DEPTHS))
  results.append(
    {"user": user, "new artists": len(findable), "setting": "most popular"}
    | found(baseline, findable)
    | {"median fans in top 10": np.median([test_fans[test_column_of[artist]] for artist in baseline[:10]])}
  )
  user_profile_changes = profile_changes_without_newest(user) if uses_tags else None
  for popularity, heavy_users, tag_weight in TEST_SETTINGS:
    picks = recommend(
      sorted(known),
      test_matrix,
      test_names,
      test_column_of,
      count=max(TEST_DEPTHS),
      popularity=popularity,
      heavy_users=heavy_users,
      tag_weight=tag_weight,
      profile_changes=user_profile_changes,
    )
    results.append(
      {"user": user, "new artists": len(findable), "setting": setting_name(popularity, heavy_users, tag_weight)}
      | found(list(picks["artist"]), findable)
      | {"median fans in top 10": picks["fans"][:10].median()}
    )

test = pd.DataFrame(results)
print(f"{test['user'].nunique()} users tested")
summary = test.groupby("setting", sort=False)[[f"top {depth}" for depth in TEST_DEPTHS] + ["median fans in top 10"]].mean()
for depth in TEST_DEPTHS:
  summary[f"top {depth}"] = (summary[f"top {depth}"] * 100).round(1).astype(str) + "%"
summary["median fans in top 10"] = summary["median fans in top 10"].round(1)
summary

428 users tested


,top 10,top 50,top 200,median fans in top 10
setting,,,,
most popular,0.6%,2.2%,6.4%,173.4
"popularity 0.5, heavy users 0.0",1.5%,4.0%,10.2%,147.1
"popularity 0.5, heavy users 0.5",1.3%,4.1%,10.3%,150.1
"popularity 0.75, heavy users 0.5",0.8%,3.5%,10.5%,79.4
"popularity 0.75, heavy users 0.5, tags 0.25",1.0%,3.9%,10.1%,92.8
"popularity 0.75, heavy users 0.5, tags 0.5",1.0%,3.7%,9.2%,90.5
"popularity 0.75, heavy users 0.5, tags 0.75",0.9%,3.4%,8.5%,87.9
"popularity 0.75, heavy users 0.5, tags 1.0",0.9%,3.2%,8.1%,85.2


In [23]:
# each user at the current settings, if the test ran them
current = test[test["setting"] == setting_name(POPULARITY_DAMPING, HEAVY_USER_DAMPING, TAG_WEIGHT)]
current.sort_values("new artists", ascending=False).round(3)

,user,new artists,setting,top 10,top 50,top 200,median fans in top 10
2475,fred2266,546,"popularity 0.75, heavy users 0.5",0.009,0.026,0.070,88.5
1803,TenedorPeligroso,541,"popularity 0.75, heavy users 0.5",0.004,0.017,0.046,126.0
219,Bhoo,539,"popularity 0.75, heavy users 0.5",0.009,0.020,0.072,129.5
1291,Naughtynaughty,530,"popularity 0.75, heavy users 0.5",0.002,0.015,0.042,129.5
2019,YamiProject,529,"popularity 0.75, heavy users 0.5",0.002,0.021,0.051,128.5
...,...,...,...,...,...,...,...
115,Aldi-Mann,1,"popularity 0.75, heavy users 0.5",0.000,0.000,0.000,4.0
1115,M1chael529,1,"popularity 0.75, heavy users 0.5",0.000,0.000,1.000,132.5
2411,electricyt,1,"popularity 0.75, heavy users 0.5",0.000,0.000,0.000,3.0
3059,respawnrookie,1,"popularity 0.75, heavy users 0.5",0.000,0.000,0.000,5.0


## Do niche users want more damping?

If people who like less popular artists do better with stronger popularity damping, the app could start each user at a
damping picked from their own likes. Users are split into thirds by how niche their earlier artists are (the median fan
count of those artists), and each third gets the test's results at every setting. If the best setting moves toward more
damping from the mainstream third to the niche third, starting from someone's likes is worth it.

How niche someone looks is tangled with how many artists they know: people with thousands of artists naturally reach
further into small ones, so the artist count is shown too.

In [24]:
from scipy import stats


def median_fans_of_earlier(user: str) -> float:
  earlier, _ = split(favourites_by_user[user])
  counts = [fan_counts[column_of[artist]] for artist in artists_of(earlier) if artist in column_of]
  return float(np.median(counts)) if counts else np.nan


tested_users = test["user"].unique()
taste = pd.DataFrame({"user": tested_users})
taste["median fans of their artists"] = taste["user"].map(median_fans_of_earlier)
taste["artists they know"] = taste["user"].map(lambda user: len(artists_of(split(favourites_by_user[user])[0])))
taste["taste"] = pd.qcut(taste["median fans of their artists"], 3, labels=["niche", "middle", "mainstream"])

by_taste = test.merge(taste, on="user")
depth = "top 200"
table = by_taste.pivot_table(index="setting", columns="taste", values=depth, aggfunc="mean", observed=True, sort=False)
print(f"share of new artists found in the {depth}, averaged within each third of users")
display(HTML((table * 100).round(1).astype(str).add("%").to_html()))
print("the setting that did best for each third:", {third: table[third].idxmax() for third in table.columns})
display(HTML(taste.groupby("taste", observed=True)[["median fans of their artists", "artists they know"]].median().round(1).to_html()))

# per user: does stronger damping help more the more niche they are?
gain = by_taste.pivot_table(index="user", columns="setting", values=depth, aggfunc="mean")
gain = (gain["popularity 0.75, heavy users 0.5"] - gain["popularity 0.5, heavy users 0.5"]).rename("gain")
paired = taste.set_index("user").join(gain)
for column in ["median fans of their artists", "artists they know"]:
  correlation, p_value = stats.spearmanr(paired[column], paired["gain"], nan_policy="omit")
  print(f"gain from popularity 0.75 over 0.5 vs {column}: spearman {correlation:+.2f} (p = {p_value:.3f})")

share of new artists found in the top 200, averaged within each third of users


taste,mainstream,niche,middle
setting,,,
most popular,11.3%,3.1%,5.6%
"popularity 0.5, heavy users 0.0",15.3%,7.0%,8.9%
"popularity 0.5, heavy users 0.5",15.7%,7.1%,8.6%
"popularity 0.75, heavy users 0.5",16.9%,6.3%,9.1%
"popularity 0.75, heavy users 0.5, tags 0.25",14.3%,7.6%,9.0%
"popularity 0.75, heavy users 0.5, tags 0.5",13.0%,7.0%,7.9%
"popularity 0.75, heavy users 0.5, tags 0.75",11.9%,6.8%,7.2%
"popularity 0.75, heavy users 0.5, tags 1.0",11.2%,6.6%,6.9%


the setting that did best for each third: {'mainstream': 'popularity 0.75, heavy users 0.5', 'niche': 'popularity 0.75, heavy users 0.5, tags 0.25', 'middle': 'popularity 0.75, heavy users 0.5'}


,median fans of their artists,artists they know
taste,,
niche,13.0,1760.0
middle,20.0,1593.5
mainstream,32.0,311.0


gain from popularity 0.75 over 0.5 vs median fans of their artists: spearman +0.21 (p = 0.000)
gain from popularity 0.75 over 0.5 vs artists they know: spearman +0.04 (p = 0.446)
